# SCD on ECG5000 — Multi-Seed Robustness Extension

Companion notebook to the *Sequential Collapse Diagnostic (SCD)* reproduction package
(Notebook 12, repository version 4).

This notebook extends the single-seed ECG5000 experiment (Notebook 10) into a **30-seed
robustness study** (seeds 42–71). The **seed-42 run reproduces the published single-seed
results exactly** (`R_ratio = 0.414 / 0.593 / 0.589`); the measurement pipeline — autoencoder,
per-series normalisation, K-means (`K=4`, `n_init=10`), and the SCD metric formulas — is
unchanged from Notebook 10. Only a per-seed reseed wrapper, robust file discovery, aggregation,
and plotting were added.

**Outputs (written to `results/`):** `ecg5000_multiseed_per_seed.csv`,
`ecg5000_multiseed_aggregate.csv`, `ecg5000_multiseed_table.csv`, `table_multiseed.tex`,
`effect_vs_noise.json`, and `Figure_12_ecg5000_multiseed.{pdf,png}`.

Run on CPU (Accelerator: None) to match the published environment.

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy import stats

DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, os.cpu_count() or 1))
LATENT_DIMS = [8, 16, 32]
EPOCHS = 20
BATCH_SIZE = 64
LR = 1e-3
HIDDEN = 64
K_CLUSTERS = 4
N_INIT = 10
EFFRANK_TOL = 0.01
N_SEEDS = 30
SEED_START = 42
SEEDS = [SEED_START + i for i in range(N_SEEDS)]
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
PUBLISHED = {8: (0.414, 8, 68.0), 16: (0.593, 16, 50.8), 32: (0.589, 20, 38.7)}

def find_file(name):
    for root in ["data", "/kaggle/input", "/kaggle/working", "."]:
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return sorted(hits)[0]
    raise FileNotFoundError(name + " not found. Place it under data/ or attach the dataset.")

def load_ts(path):
    X, y = [], []
    started = False
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if s.lower().startswith("@data"):
                started = True
                continue
            if s.startswith("@") or s.startswith("#"):
                continue
            if not started:
                continue
            if ":" in s:
                series, label = s.rsplit(":", 1)
            else:
                series, label = s, "0"
            X.append([float(v) for v in series.split(",") if v.strip() != ""])
            y.append(label.strip())
    return np.array(X, dtype=float), np.array(y)

train_path = find_file("ECG5000_TRAIN.ts")
test_path = find_file("ECG5000_TEST.ts")
print("TRAIN:", train_path)
print("TEST :", test_path)
X_train, y_train = load_ts(train_path)
X_test, y_test = load_ts(test_path)
X = np.vstack([X_train, X_test])
y = np.hstack([y_train, y_test])
print("Dataset:", X.shape[0], "series x", X.shape[1], "points | classes:", np.unique(y))
X_norm = np.zeros_like(X)
for i in range(X.shape[0]):
    X_norm[i] = (X[i] - X[i].mean()) / (X[i].std() + 1e-8)

class TimeSeriesAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, input_dim))
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

def train_ae(Xn, latent_dim):
    dataset = TensorDataset(torch.tensor(Xn.astype(np.float32)))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    model = TimeSeriesAE(Xn.shape[1], latent_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    model.train()
    for epoch in range(EPOCHS):
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            optimizer.zero_grad()
            recon, _ = model(xb)
            loss = criterion(recon, xb)
            loss.backward()
            optimizer.step()
    return model

def compute_scd(Z, seed):
    km = KMeans(n_clusters=K_CLUSTERS, random_state=seed, n_init=N_INIT)
    labels = km.fit_predict(Z)
    runs = 1 + np.sum(labels[:-1] != labels[1:])
    n_k = np.bincount(labels, minlength=K_CLUSTERS)
    N = len(labels)
    R_null = 1 + (N * N - np.sum(n_k * n_k)) / N
    R_ratio = runs / R_null
    _, S, _ = np.linalg.svd(Z, full_matrices=False)
    eff_rank = int(np.sum(S > EFFRANK_TOL * S[0]))
    pca = PCA(n_components=1); pca.fit(Z)
    pc1 = float(pca.explained_variance_ratio_[0] * 100)
    return float(R_ratio), eff_rank, pc1, bool(R_ratio < 0.5)

records = []
for seed in SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)
    for ld in LATENT_DIMS:
        model = train_ae(X_norm, ld)
        model.eval()
        with torch.no_grad():
            _, Z = model(torch.tensor(X_norm.astype(np.float32)).to(DEVICE))
        Z = Z.cpu().numpy()
        r, er, pc1, viol = compute_scd(Z, seed)
        records.append({"seed": seed, "latent_dim": ld, "R_ratio": r, "effrank": er, "PC1_var": pc1, "NDC6_violated": viol})
    print("seed", seed, "ok")

df = pd.DataFrame(records)
df.to_csv(os.path.join(RESULTS_DIR, "ecg5000_multiseed_per_seed.csv"), index=False)

def cv(a):
    mu = np.mean(a)
    return float(np.std(a, ddof=1) / abs(mu) * 100) if mu != 0 else float("nan")

rows = []
for ld in LATENT_DIMS:
    g = df[df.latent_dim == ld]
    row = {"latent_dim": ld}
    for col in ["R_ratio", "effrank", "PC1_var"]:
        v = g[col].values.astype(float)
        row[col + "_mean"] = float(v.mean()); row[col + "_sd"] = float(v.std(ddof=1))
        row[col + "_min"] = float(v.min()); row[col + "_max"] = float(v.max()); row[col + "_cv"] = cv(v)
    viol = g["NDC6_violated"].mean()
    row["NDC6_rate"] = float(viol)
    row["NDC6_decision"] = "Collapse (violated)" if viol == 1.0 else "Non-collapse" if viol == 0.0 else "UNSTABLE %d%%" % round(viol * 100)
    rows.append(row)
agg = pd.DataFrame(rows)
agg.to_csv(os.path.join(RESULTS_DIR, "ecg5000_multiseed_aggregate.csv"), index=False)

pub = pd.DataFrame({
    "latent_dim": agg.latent_dim,
    "R_ratio_mean_sd": ["%.3f +/- %.3f" % (m, s) for m, s in zip(agg.R_ratio_mean, agg.R_ratio_sd)],
    "R_ratio_min_max": ["[%.3f, %.3f]" % (a, b) for a, b in zip(agg.R_ratio_min, agg.R_ratio_max)],
    "R_ratio_CV_pct": ["%.1f" % c for c in agg.R_ratio_cv],
    "effrank_mean_sd": ["%.1f +/- %.1f" % (m, s) for m, s in zip(agg.effrank_mean, agg.effrank_sd)],
    "PC1_mean_sd": ["%.1f +/- %.1f" % (m, s) for m, s in zip(agg.PC1_var_mean, agg.PC1_var_sd)],
    "NDC6_violation_rate": ["%.0f%%" % (r * 100) for r in agg.NDC6_rate],
})
pub.to_csv(os.path.join(RESULTS_DIR, "ecg5000_multiseed_table.csv"), index=False)
print("\\n=== Multi-seed robustness table ===")
print(pub.to_string(index=False))

tex = ["\\\\begin{table}[ht]\\\\centering",
       "\\\\caption{Multi-seed robustness of the SCD on ECG5000 (30 seeds, 42--71). Mean$\\\\pm$SD across seeds.}",
       "\\\\label{tab:ecg5000_multiseed}\\\\begin{tabular*}{\\\\tblwidth}{@{}lccccc@{}}\\\\toprule",
       "$d_{\\\\mathrm{model}}$ & $R_{\\\\mathrm{ratio}}$ (mean$\\\\pm$SD) & [min, max] & CV(\\\\%) & effrank & PC1 (\\\\%) \\\\\\\\\\\\midrule"]
for _, r in agg.iterrows():
    tex.append("%d & %.3f$\\\\pm$%.3f & [%.3f, %.3f] & %.1f & %.1f$\\\\pm$%.1f & %.1f$\\\\pm$%.1f \\\\\\\\" % (
        int(r.latent_dim), r.R_ratio_mean, r.R_ratio_sd, r.R_ratio_min, r.R_ratio_max, r.R_ratio_cv,
        r.effrank_mean, r.effrank_sd, r.PC1_var_mean, r.PC1_var_sd))
tex += ["\\\\bottomrule\\\\end{tabular*}\\\\end{table}"]
open(os.path.join(RESULTS_DIR, "table_multiseed.tex"), "w").write("\\n".join(tex))

r8 = df[df.latent_dim == 8]["R_ratio"].values
r16 = df[df.latent_dim == 16]["R_ratio"].values
psd = np.sqrt(((len(r8)-1)*r8.var(ddof=1) + (len(r16)-1)*r16.var(ddof=1)) / (len(r8)+len(r16)-2))
cohen = (r16.mean()-r8.mean())/psd if psd > 0 else float("inf")
U, pmw = stats.mannwhitneyu(r8, r16, alternative="two-sided")
json.dump({"cohen_d": float(cohen), "mannwhitney_p": float(pmw),
           "ranges_disjoint": bool(r8.max() < r16.min())},
          open(os.path.join(RESULTS_DIR, "effect_vs_noise.json"), "w"), indent=2)

CM = 1/2.54; DOUBLE = 17.8*CM
plt.rcParams.update({
    "pdf.fonttype":42, "ps.fonttype":42,
    "font.family":"sans-serif", "font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "font.size":8, "axes.titlesize":9, "axes.labelsize":8,
    "xtick.labelsize":7, "ytick.labelsize":7, "legend.fontsize":7,
    "axes.linewidth":0.8, "lines.linewidth":1.4, "lines.markersize":5,
    "axes.grid":True, "grid.alpha":0.3, "grid.linewidth":0.5,
    "legend.frameon":True, "legend.framealpha":0.9,
    "savefig.bbox":"tight", "savefig.pad_inches":0.02, "figure.dpi":120,
})
mm = agg.set_index("latent_dim")
fig, axes = plt.subplots(1, 3, figsize=(DOUBLE, 6.4*CM))
ax = axes[0]
data = [df[df.latent_dim == ld]["R_ratio"].values for ld in LATENT_DIMS]
bp = ax.boxplot(data, positions=range(3), widths=0.55, patch_artist=True, showfliers=False)
for p in bp["boxes"]: p.set(facecolor="#9ecae1", alpha=0.7, linewidth=0.8)
for grp in ("whiskers","caps","medians"):
    for ln in bp[grp]: ln.set_linewidth(0.8)
jit = np.random.default_rng(0)
for i, d in enumerate(data):
    ax.scatter(np.full_like(d, i)+jit.uniform(-0.13,0.13,len(d)), d, s=9, color="#08519c", alpha=0.65, zorder=3, linewidths=0)
ax.axhline(0.5, color="red", ls="--", lw=1.2, label="NDC-6 threshold")
ax.set_xticks(range(3)); ax.set_xticklabels(LATENT_DIMS)
ax.set_xlabel("Latent dimension"); ax.set_ylabel(r"$R_{\mathrm{ratio}}$")
ax.legend(loc="lower right"); ax.set_title("(a) Per-seed $R_{\\mathrm{ratio}}$", loc="left")
ax = axes[1]
ax.errorbar(LATENT_DIMS, mm.effrank_mean, yerr=mm.effrank_sd, fmt="s-", lw=1.4, ms=5, capsize=3, color="#2c7a3f")
ax.set_xticks(LATENT_DIMS); ax.set_xlabel("Latent dimension"); ax.set_ylabel("Effective rank")
ax.set_title("(b) Effective rank", loc="left")
ax = axes[2]
ax.errorbar(LATENT_DIMS, mm.PC1_var_mean, yerr=mm.PC1_var_sd, fmt="^-", lw=1.4, ms=5, capsize=3, color="#a6611a")
ax.set_xticks(LATENT_DIMS); ax.set_xlabel("Latent dimension"); ax.set_ylabel("PC1 variance (%)")
ax.set_title("(c) PC1 variance", loc="left")
fig.tight_layout(pad=0.4)
fig.savefig(os.path.join(RESULTS_DIR, "Figure_12_ecg5000_multiseed.pdf"))
fig.savefig(os.path.join(RESULTS_DIR, "Figure_12_ecg5000_multiseed.png"), dpi=600)
fig.savefig(os.path.join(RESULTS_DIR, "Figure_12_ecg5000_multiseed.tiff"), dpi=600)
print("\\nSaved Figure_12 (pdf + png).")

ref = df[df.seed == SEED_START].set_index("latent_dim")
print("\\n=== Reproducibility check (seed 42 vs published) ===")
for ld in LATENT_DIMS:
    g = ref.loc[ld]; p = PUBLISHED[ld]
    print("ld=%2d  R_ratio %.3f (pub %.3f) | effrank %d (pub %d) | PC1 %.1f (pub %.1f)" %
          (ld, g.R_ratio, p[0], int(g.effrank), p[1], g.PC1_var, p[2]))
print("\\nEffect (ld8 vs ld16): Cohen d = %.2f | Mann-Whitney p = %.2e" % (cohen, pmw))
print("All outputs written to:", os.path.abspath(RESULTS_DIR))
